In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from spx_history import *
from intra_adj import *

- for one symbol first

In [3]:
start_dt = dt.date(2021,1,1)
end_dt = dt.date(2026,2,1)

- one symbol only

In [4]:
# all dividends data
ALL_DIV = pd.read_csv('data_by_date/all_dividends_till_20260201.csv')
ALL_DIV['Date'] = pd.to_datetime(ALL_DIV['Date']).dt.date

# read holidays
th_us = pd.read_csv('us_holidays.csv')
th_us['date'] = pd.to_datetime(th_us['date'], dayfirst=True).dt.date
US_HOLS = th_us['date'].unique().tolist()

In [5]:
symbol = "AMZN"

In [ ]:
ti = get_adjusted_intraday_by_sym(symbol=symbol, all_div=ALL_DIV, hols=US_HOLS, keep_unadj=True)

In [ ]:
all_div = ALL_DIV
hols = US_HOLS
keep_unadj = True

In [ ]:
ti["ret_1m"] = 100* np.log(ti["close"] / ti.groupby("date")["close"].shift(1)).fillna(0)

In [ ]:
ti = ti[
	(ti['time'] >= dt.time(9, 30)) &
	(ti['time'] <= dt.time(16, 0))
]

In [ ]:
sdate = dt.date(2025, 12, 24)

In [ ]:
ti.groupby('date').agg(
	mean_ret = ("ret_1m", "mean"),
	std_ret = ("ret_1m", "std"),
	min_ret = ("ret_1m", "min"),
	max_ret = ("ret_1m", "max"),
	count_zero_returns = ("ret_1m", lambda x: (x==0).sum()),
	count_all = ("ret_1m", "count")
).plot(y=['mean_ret', 'max_ret', 'min_ret', 'std_ret'])

- adjusted intraday history

In [7]:
tu = pd.read_csv('data/spy_cst.csv')
tickers = tu[tu['Ticker'] != '-']['Ticker'].unique().tolist()

wts = pd.read_csv('spy_weight_historical.csv')
wts_tickers_5y = wts['ticker'].unique().tolist()
rem_tickers = [x for x in wts_tickers_5y if not x in tickers]

In [9]:
# [x for x in tickers if not x in wts_tickers_5y]

In [15]:
for sym in rem_tickers:
	
	try:
		qt.log.info(f"querying for ticker : {sym}")

		csv_file_name = f'data_adjusted_by_symbol/{sym}.csv'
		if Path(csv_file_name).is_file():
			qt.log.info(f"file already exists. ignoring")
		else:
			ti_sym = get_adjusted_intraday_by_sym(symbol=sym, all_div=ALL_DIV, hols=US_HOLS, keep_unadj=True)
			qt.log.info(f"got {ti_sym.shape} rows for sym {sym}, saving to {csv_file_name}")
			ti_sym.to_csv(csv_file_name, index=False)
	
	except Exception as e:
		qt.log.warning(f"failed for ticker {sym}, ereor : {e}")

[JUSTY.LOG]	2026-02-21 00:59:57,982 - qt.common.help - INFO - querying for ticker : FB
[JUSTY.LOG]	2026-02-21 00:59:57,983 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-21 00:59:57,984 - qt.common.help - INFO - querying for ticker : BRKB
[JUSTY.LOG]	2026-02-21 00:59:57,985 - qt.common.help - WARNING - split file not found for BRKB
[JUSTY.LOG]	2026-02-21 00:59:57,985 - qt.common.help - INFO - df.shape = (0, 0)
[JUSTY.LOG]	2026-02-21 00:59:57,986 - qt.common.help - WARNING - intraday file not found for BRKB
[JUSTY.LOG]	2026-02-21 00:59:57,986 - qt.common.help - INFO - df.shape = (0, 0)
[JUSTY.LOG]	2026-02-21 00:59:57,987 - qt.common.help - WARNING - failed for ticker BRKB, ereor : 'timestamp'
[JUSTY.LOG]	2026-02-21 00:59:57,987 - qt.common.help - INFO - querying for ticker : ANTM
[JUSTY.LOG]	2026-02-21 00:59:57,987 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-21 00:59:57,988 - qt.common.help - INFO - querying for ticker : ATVI


In [ ]:
ti_sym.groupby('date')['close'].apply(lambda x: (~x.isna()).sum())

In [ ]:
ti_sym.groupby('date')['close'].apply(lambda x: x.count())

In [ ]:
ti_sym[ti_sym['']]